# KG → LLM Fine-Tuning Evaluation Pipeline

This notebook runs the full evaluation pipeline on **Google Colab with a T4 GPU**. Open in Colab, fill in your configuration below, then run all cells.

| Method | What it does | GPU needed? |
|--------|-------------|-------------|
| **Method 1** — SFT Data Quality | Structural audit, SFT pair generation, quality scoring | No (CPU only) |
| **Method 2** — Ablation Study | Fine-tune Model B (KG-structured) & Model C (raw-text), benchmark vs base Model A | **Yes (T4 GPU)** |

### Prerequisites
- A knowledge graph uploaded to Neo4j (`kg-gen neo4j-upload` from your local machine)
- Neo4j credentials (AuraDB free tier works great)
- DeepSeek API key for SFT pair generation

## ⚙️ Configuration

Set your values in the cell below, then run it. All subsequent cells will use these values.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — Fill in your values below
# ═══════════════════════════════════════════════════════════════
#
# ⚠️  This notebook will contain secrets after you fill it in.
#     Do NOT commit your filled-in copy to git.

import os

# ── Repository URL ───────────────────────────────────────────
REPO_URL = "https://github.com/your-org/hackathon.git"  # ⬅️ REPLACE ME

# ── Neo4j — KG database connection ───────────────────────────
NEO4J_URI = "bolt://your-instance.databases.neo4j.io:7687"  # ⬅️ REPLACE ME
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "your-password"  # ⬅️ REPLACE ME

# ── DeepSeek — for LLM-powered SFT pair generation ───────────
# Get your key at: https://platform.deepseek.com/api_keys
DEEPSEEK_API_KEY = "sk-..."  # ⬅️ REPLACE ME

# ── Model — override the base model for fine-tuning (optional) ─
# Default: Qwen/Qwen2.5-1.5B-Instruct (~2 GB VRAM in 4-bit)
# Alternatives: Qwen/Qwen2.5-0.5B-Instruct (fastest), Qwen/Qwen2.5-3B-Instruct (better)
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

print("✅ Configuration variables set.")
print(f"   Repo:           {REPO_URL}")
print(f"   Neo4j URI:      {NEO4J_URI}")
print(f"   Neo4j user:     {NEO4J_USER}")
print(f"   DeepSeek key:   {'***' + DEEPSEEK_API_KEY[-4:] if len(DEEPSEEK_API_KEY) > 4 else 'NOT SET'}")
print(f"   Base model:     {BASE_MODEL}")
print(f"\n⬆️  Edit the values marked 'REPLACE ME' above before continuing.")

## Step 1: Check GPU

Make sure a GPU is available for Method 2 (fine-tuning). Method 1 runs fine on CPU.

In [ ]:
import torch
import subprocess
import sys

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU available: {gpu_name} ({vram:.1f} GB VRAM)")
else:
    print("⚠️  No GPU detected — Method 2 (fine-tuning) will NOT work.")
    print("   Go to Runtime → Change runtime type → select T4 GPU.")

## Step 2: Clone the Repository & Install Dependencies

In [ ]:
# ═══ Clone the repo ═══
!git clone {REPO_URL}
%cd hackathon

# ═══ Install dependencies ═══
# eval-model: transformers, peft, accelerate, bitsandbytes (for fine-tuning)
# neo4j: Neo4j Python driver (for downloading the KG)
!pip install -e ".[eval-model,neo4j]"

print("\n✅ Dependencies installed.")
print("   If prompted to restart runtime: Runtime → Restart runtime, then re-run from the Configuration cell.")

## Step 3: Set Credentials

Set your **Neo4j connection details** (to download the KG) and **DeepSeek API key** (for SFT pair generation).

> ⚠️ **Never commit secrets to git.** These are set as runtime environment variables only.

In [ ]:
import os

# ═══ Set credentials from the Configuration cell above ═══
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USER"] = NEO4J_USER
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD
os.environ["DEEPSEEK_API_KEY"] = DEEPSEEK_API_KEY

# ═══ Write a local .env so kg-gen picks it up ═══
with open(".env", "w") as f:
    f.write(f'NEO4J_URI={os.environ["NEO4J_URI"]}\n')
    f.write(f'NEO4J_USER={os.environ["NEO4J_USER"]}\n')
    f.write(f'NEO4J_PASSWORD={os.environ["NEO4J_PASSWORD"]}\n')
    f.write(f'DEEPSEEK_API_KEY={os.environ["DEEPSEEK_API_KEY"]}\n')

print("✅ Credentials set and .env written.")
print(f"   Neo4j URI: {os.environ['NEO4J_URI']}")
print(f"   DeepSeek key: ***{os.environ['DEEPSEEK_API_KEY'][-4:]}")

## Step 4: Download the Knowledge Graph from Neo4j

Pulls the full graph (nodes + relationships) from Neo4j and saves it as `knowledge_graph.json` — the format expected by the evaluation pipeline.

In [ ]:
import subprocess, os, json
from pathlib import Path

# Ensure output directory exists
output_dir = Path("generated_KGs/output")
output_dir.mkdir(parents=True, exist_ok=True)
kg_path = output_dir / "knowledge_graph.json"

# Download from Neo4j
print("Downloading knowledge graph from Neo4j...")
result = subprocess.run(
    ["kg-gen", "neo4j-download", "-o", str(kg_path)],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ Download failed:")
    print(result.stderr)
    print("\nTroubleshooting:")
    print("  - Is your Neo4j instance running and accessible?")
    print("  - Did you set NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD correctly?")
    print("  - Did you upload the KG first? (make upload on your local machine)")
    raise SystemExit(1)

print(result.stdout)

# Quick sanity check
if kg_path.exists():
    with open(kg_path) as f:
        data = json.load(f)
    stats = data.get("stats", {})
    print(f"\n✅ KG downloaded successfully!")
    print(f"   Nodes: {stats.get('num_nodes', '?'):,}")
    print(f"   Edges: {stats.get('num_edges', '?'):,}")
    print(f"   Triples: {stats.get('num_triples', '?'):,}")
    print(f"   File size: {kg_path.stat().st_size / 1e6:.1f} MB")
else:
    print("❌ knowledge_graph.json not found after download.")
    raise FileNotFoundError(str(kg_path))

## Step 5: Method 1 — SFT Data Quality Assessment

Runs on CPU. Checks graph health, generates SFT training pairs, and evaluates their quality.

**⏱ ~30 seconds**

In [ ]:
import subprocess, json
from pathlib import Path

kg_path = "generated_KGs/output/knowledge_graph.json"

print("=" * 60)
print("METHOD 1: SFT Data Quality Assessment")
print("=" * 60)

result = subprocess.run(
    ["python", "evaluation/run_eval.py", "--method", "1", "--kg", kg_path],
    capture_output=True, text=True
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:2000])

# Show key results from the output files
method1_dir = Path("output_eval/latest/method1")

# Structural audit summary
audit_path = method1_dir / "structural_audit.json"
if audit_path.exists():
    with open(audit_path) as f:
        audit = json.load(f)
    print(f"\n📊 Structural Audit — Health Score: {audit.get('overall_health_score', '?')}/100")
    print(f"   Verdict: {audit.get('verdict', '?')}")

# SFT quality summary
quality_path = method1_dir / "sft_quality_report.json"
if quality_path.exists():
    with open(quality_path) as f:
        quality = json.load(f)
    print(f"\n📊 SFT Quality — Overall Score: {quality.get('overall_score', '?'):.3f}")
    print(f"   Verdict: {quality.get('verdict', '?')}")

print(f"\n✅ Method 1 complete — full results in {method1_dir}/")

## Step 6: Method 2 — Fine-Tune Both Models

Fine-tunes Qwen2.5 on **two** training datasets, then benchmarks all three models:

| Model | Training Data | Hypothesis |
|-------|--------------|------------|
| **A** | None (base Qwen2.5) | Baseline |
| **B** | KG-structured QA pairs | KG improves multi-hop reasoning |
| **C** | Raw-text QA pairs | Same facts, no graph structure |

This cell generates QA datasets, trains Model B and Model C, and runs the benchmark.

**⏱ ~25–35 minutes on T4 GPU**

In [ ]:
import subprocess

kg_path = "generated_KGs/output/knowledge_graph.json"

print("=" * 60)
print("METHOD 2: Fine-Tuning + Benchmark")
print("=" * 60)
print(f"Base model: {BASE_MODEL}")
print("This will: generate QA datasets → train Model B (KG) → train Model C (raw) → benchmark")
print("Expected time: ~25-35 min on T4 GPU\n")

result = subprocess.run(
    [
        "python", "evaluation/run_eval.py",
        "--method", "2",
        "--kg", kg_path,
        "--fine-tune-target", "both",
        "--model", BASE_MODEL,
    ],
    capture_output=False,  # stream to notebook output
    text=True
)

if result.returncode == 0:
    print("\n✅ Method 2 complete!")
    print("   Results saved to output_eval/latest/method2/")
else:
    print(f"\n❌ Method 2 failed with exit code {result.returncode}")
    print("   Common issues:")
    print("   - Out of VRAM → switch to a smaller model (edit BASE_MODEL in Configuration)")
    print("   - Missing dependencies → re-run Step 2")

## Step 7: Download Results

After everything completes, zip and download the results folder.

In [ ]:
import shutil
from pathlib import Path

# Zip the results
output_zip = "evaluation_results.zip"
shutil.make_archive("evaluation_results", "zip", "output_eval")

print(f"✅ Results zipped to {output_zip}")
print(f"   Size: {Path(output_zip).stat().st_size / 1e6:.1f} MB")

# Download to your local machine
from google.colab import files
files.download(output_zip)

# Also list individual files
print("\nYou can also browse individual files:")
for f in sorted(Path("output_eval").rglob("*.json")):
    print(f"   output_eval/{f.relative_to('output_eval')}")

## Troubleshooting

<details>
<summary><b>❌ "No module named 'kg_generator'"</b></summary>

Run `!pip install -e ".[eval-model,neo4j]"` again — the package may not be installed in editable mode.
</details>

<details>
<summary><b>❌ Neo4j connection refused</b></summary>

- Make sure your Neo4j instance is running and accessible from the internet.
- AuraDB free tier: check the instance is active in the Neo4j Aura Console.
- Self-hosted: ensure port 7687 is open. Consider using AuraDB for simplicity.
</details>

<details>
<summary><b>❌ CUDA out of memory during fine-tuning</b></summary>

- Switch to a smaller model: edit `BASE_MODEL` in the Configuration cell to `"Qwen/Qwen2.5-0.5B-Instruct"`.
- Or use a 4-bit quantized model: `"unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"`.
- Reduce batch size: edit `eval_config.yaml` → `training.per_device_train_batch_size: 1`.
</details>

<details>
<summary><b>❌ DeepSeek API key not working</b></summary>

- Verify your key at https://platform.deepseek.com/api_keys
- The SFT generator falls back to template-based generation without an API key (lower quality but still functional).
</details>

<details>
<summary><b>❌ KG file not found after download</b></summary>

- Make sure you uploaded the KG to Neo4j first: run `kg-gen neo4j-upload` on your local machine.
- Check that your NEO4J_URI, NEO4J_USER, and NEO4J_PASSWORD are correct in the Configuration cell.
</details>